# Qwen2.5-7B-Instruct LoRA Fine-tuning for Robot Task Planning

**Pipeline:**
1. Load dataset (ChatML format)
2. Load Qwen2.5-7B-Instruct in 4-bit
3. Apply LoRA
4. Train with SFTTrainer (trl)
5. Merge LoRA into base weights
6. Save merged model
7. Instructions for GGUF conversion and Ollama import

**Requirements:** GPU must be enabled. Use T4 on Kaggle (free tier).

In [1]:
# Verify GPU is available FIRST. If this fails, enable GPU in Kaggle settings.
import torch
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE'}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB" if torch.cuda.is_available() else "")
assert torch.cuda.is_available(), "Enable GPU in Kaggle settings before running this notebook."

CUDA available: True
GPU: Tesla T4
VRAM: 15.6 GB


In [2]:
!pip install -q transformers datasets peft accelerate bitsandbytes trl
!pip install -U trl transformers accelerate

In [3]:
import trl
print(trl.__version__)

0.29.1


In [4]:
import json
import torch
from datasets import Dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model
from trl import SFTTrainer, SFTConfig

MODEL_NAME = "Qwen/Qwen2.5-7B-Instruct"
DATASET_PATH = "/kaggle/input/datasets/chaitanyaparate/ollama-robot-fine-tune/dataset.json"  # update this path
OUTPUT_DIR = "/kaggle/working/qwen_robot_lora"
MERGED_DIR = "/kaggle/working/qwen_robot_merged"

print(f"Model: {MODEL_NAME}")
print(f"Output: {OUTPUT_DIR}")

Model: Qwen/Qwen2.5-7B-Instruct
Output: /kaggle/working/qwen_robot_lora


## 1. Load Dataset

In [5]:
with open(DATASET_PATH, "r") as f:
    raw = json.load(f)

dataset = Dataset.from_list(raw)

# Quick sanity checks
print(f"Total samples: {len(dataset)}")
print(f"Columns: {dataset.column_names}")
print("\n--- First sample ---")
print(dataset[0]["text"][:500])

assert "text" in dataset.column_names, "Dataset must have a 'text' column in ChatML format."

Total samples: 3000
Columns: ['text']

--- First sample ---
<|im_start|>system
You are a robot task planner. Given a natural language task and a scene description, produce a structured JSON action plan. The plan must be a list of skill steps the robot will execute in order. Each step must have: skill (string), target (string), constraints (dict with 'avoid' list). Always respond with valid JSON only. No explanation, no markdown.<|im_end|>
<|im_start|>user
Task: First navigate the orange_cone, then navigate the yellow_cylinder. Avoid nothing.
Scene: {"obj


## 2. Load Tokenizer

In [6]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)

# Qwen2.5 uses <|endoftext|> as pad token; set explicitly to avoid warnings
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"  # Required for SFTTrainer

print(f"Vocab size: {tokenizer.vocab_size}")
print(f"Pad token: {tokenizer.pad_token}")

Vocab size: 151643
Pad token: <|endoftext|>


## 3. Load Model in 4-bit

In [7]:
from transformers import BitsAndBytesConfig, AutoModelForCausalLM
import torch

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,  # compute dtype
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4"
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    torch_dtype=torch.float16,   # MUST be here
    device_map="auto",
    trust_remote_code=True
)

model.config.torch_dtype = torch.float16

# Disable cache for training (required with gradient checkpointing)
model.config.use_cache = False
model.config.pretraining_tp = 1

from peft import prepare_model_for_kbit_training
model = prepare_model_for_kbit_training(model)

dtypes = set(p.dtype for p in model.parameters())
print(dtypes)

print(f"Model loaded. Device map: {model.hf_device_map}")

`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

{torch.float32, torch.uint8}
Model loaded. Device map: {'model.embed_tokens': 0, 'model.layers.0': 0, 'model.layers.1': 0, 'model.layers.2': 0, 'model.layers.3': 1, 'model.layers.4': 1, 'model.layers.5': 1, 'model.layers.6': 1, 'model.layers.7': 1, 'model.layers.8': 1, 'model.layers.9': 1, 'model.layers.10': 1, 'model.layers.11': 1, 'model.layers.12': 1, 'model.layers.13': 1, 'model.layers.14': 1, 'model.layers.15': 1, 'model.layers.16': 1, 'model.layers.17': 1, 'model.layers.18': 1, 'model.layers.19': 1, 'model.layers.20': 1, 'model.layers.21': 1, 'model.layers.22': 1, 'model.layers.23': 1, 'model.layers.24': 1, 'model.layers.25': 1, 'model.layers.26': 1, 'model.layers.27': 1, 'model.norm': 1, 'model.rotary_emb': 1, 'lm_head': 1}


## 4. Apply LoRA

In [8]:
from peft import prepare_model_for_kbit_training

# Required step before applying LoRA to a quantized model

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    # Qwen2.5 attention projection names
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

# Expected output: trainable params ~40M (0.5%) out of 7.6B

trainable params: 40,370,176 || all params: 7,655,986,688 || trainable%: 0.5273


## 5. Train with SFTTrainer

In [ ]:
# SFTConfig replaces TrainingArguments in modern trl versions


def preprocess(example):
    return tokenizer(
        example["text"],
        truncation=True,
        max_length=256,
    )

dataset = dataset.map(preprocess)

from trl import SFTTrainer, SFTConfig

training_args = SFTConfig(
    output_dir=OUTPUT_DIR,
    num_train_epochs=3,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    dataloader_num_workers=2,
    gradient_checkpointing=True,
    optim="adamw_torch",
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_steps=50,  # ✅ fixed
    logging_steps=100,
    save_steps=200,
    save_total_limit=2,
    fp16=False,
    bf16=False,
    dataset_text_field="text",
    report_to="none",
)

trainer = SFTTrainer(
    model=model,
    train_dataset=dataset,
    args=training_args,
    processing_class=tokenizer,   # ✅ NEW (replaces tokenizer)
    
)

print(f"Training samples: {len(dataset)}")
print(f"Steps per epoch: {len(dataset) // (training_args.per_device_train_batch_size * training_args.gradient_accumulation_steps)}")
print("Starting training...")

import torch

dtypes = {}
for name, p in model.named_parameters():
    dtypes[p.dtype] = dtypes.get(p.dtype, 0) + 1

print(dtypes)

tokenizer.bos_token = tokenizer.eos_token
model.config.bos_token_id = tokenizer.eos_token_id

model = torch.compile(model)

trainer.train()
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f"LoRA adapter saved to {OUTPUT_DIR}")

Map:   0%|          | 0/3000 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/3000 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': 151645, 'pad_token_id': 151643}.


Training samples: 3000
Steps per epoch: 187
Starting training...
{torch.float32: 143, torch.uint8: 196, torch.bfloat16: 392}


Step,Training Loss


## 6. Merge LoRA into Base Model and Save

Merging is required before GGUF conversion. You cannot convert a PEFT adapter directly.

In [ ]:
from peft import PeftModel
import gc

# Clear GPU cache before merge
torch.cuda.empty_cache()
gc.collect()

print("Loading base model for merge (fp16, no quantization)...")
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map="cpu",  # Merge on CPU to avoid OOM
    trust_remote_code=True
)

print("Loading LoRA adapter...")
peft_model = PeftModel.from_pretrained(base_model, OUTPUT_DIR)

print("Merging weights...")
merged_model = peft_model.merge_and_unload()

print(f"Saving merged model to {MERGED_DIR}...")
merged_model.save_pretrained(MERGED_DIR, safe_serialization=True)
tokenizer.save_pretrained(MERGED_DIR)

print("Merge complete.")
del base_model, peft_model, merged_model
torch.cuda.empty_cache()
gc.collect()

## 7. Quick Inference Test (before export)

In [ ]:
# Test the merged model locally on Kaggle
from transformers import pipeline

test_model = AutoModelForCausalLM.from_pretrained(
    MERGED_DIR,
    
    torchdtype=torch.float16,
    device_map="auto"
)
test_tokenizer = AutoTokenizer.from_pretrained(MERGED_DIR)

messages = [
    {"role": "system", "content": "You are a robot task planner. Given a task and scene, output a JSON action plan only."},
    {"role": "user", "content": 'Task: Go to the red_box.\nScene: {"objects": ["red_box", "blue_sphere"], "obstacles": ["wall"]}'}
]

text = test_tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
inputs = test_tokenizer(text, return_tensors="pt").to(test_model.device)

with torch.no_grad():
    out = test_model.generate(**inputs, max_new_tokens=200, temperature=0.1, do_sample=True)

response = test_tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
print("Model output:")
print(response)

# Verify it's valid JSON
try:
    parsed = json.loads(response.strip())
    print("\nValid JSON output.")
except json.JSONDecodeError as e:
    print(f"\nInvalid JSON: {e} -- fine-tuning may need more epochs or dataset fixes.")

## 8. Export to GGUF and Import into Ollama (run LOCALLY, not on Kaggle)

After downloading `/kaggle/working/qwen_robot_merged/` to your machine:

```bash
# Step 1: Clone llama.cpp
git clone https://github.com/ggerganov/llama.cpp
cd llama.cpp
pip install -r requirements.txt

# Step 2: Convert merged HuggingFace model to GGUF (Q4_K_M quantization)
python convert_hf_to_gguf.py /path/to/qwen_robot_merged --outfile qwen_robot_planner.gguf --outtype q4_k_m

# Step 3: Create Ollama Modelfile
cat > Modelfile << 'EOF'
FROM ./qwen_robot_planner.gguf

SYSTEM "You are a robot task planner. Given a natural language task and a scene description, produce a structured JSON action plan. The plan must be a list of skill steps. Each step must have: skill (string), target (string), constraints (dict with avoid list). Always respond with valid JSON only. No explanation, no markdown."

PARAMETER temperature 0.1
PARAMETER top_p 0.9
PARAMETER stop "<|im_end|>"
EOF

# Step 4: Register model in Ollama
ollama create qwen-robot-planner -f Modelfile

# Step 5: Test
ollama run qwen-robot-planner 'Task: Navigate to the red_box. Scene: {"objects": ["red_box"], "obstacles": ["wall"]}'
```

In your ROS2 LLM planner node, replace the API call with:
```python
import subprocess, json

def call_llm_planner(task: str, scene: dict) -> dict:
    prompt = f'Task: {task}\nScene: {json.dumps(scene)}'
    result = subprocess.run(
        ["ollama", "run", "qwen-robot-planner", prompt],
        capture_output=True, text=True, timeout=30
    )
    return json.loads(result.stdout.strip())
```
Or use the Ollama Python library: `pip install ollama`
```python
import ollama, json

def call_llm_planner(task: str, scene: dict) -> dict:
    response = ollama.chat(
        model="qwen-robot-planner",
        messages=[{"role": "user", "content": f'Task: {task}\nScene: {json.dumps(scene)}'}]
    )
    return json.loads(response["message"]["content"].strip())
```